# Reproducing Mountoufaris et al. (2024) from dandiset 001131: activity, selectivity, decoding and persistence

Paper. Mountoufaris G, Nair A, Yang B, Kim D-W, Vinograd A, Kim S, Linderman SW, Anderson DJ (2024).
A line attractor encoding a persistent internal state requires neuropeptide signaling. *Cell* 187,
5998-6015. [doi:10.1016/j.cell.2024.08.015](https://doi.org/10.1016/j.cell.2024.08.015). The claims
file (`claims/001131/10.1016-j.cell.2024.08.015.yaml`) was written from the accepted author manuscript
in PubMed Central (PMC11490375), with the supplementary figures; this attempt read the same text.

Dandiset. [DANDI:001131](https://dandiarchive.org/dandiset/001131/0.260601.1203), version
0.260601.1203. The paper cites the dandiset without a version (Key resources table, "DANDI:001131").
Two versions are published. 0.240826.1647 holds two files, one control and one experimental mouse;
0.260601.1203, the most recent, holds ten, and its files for mice 510 and 500 hold the same data as
the two earlier files (checked below). The most recent version is pinned because it holds all the
data of the earlier one and more, and so matches the claims file's description better.

What the dandiset holds. One file per male resident: 4 control (scrambled gRNA) and 6 experimental
(Oxtr/Avpr1a gRNAs) mice, each with a z-scored trace per unit at 10 Hz and the annotated sniffing,
attack and dominance mounting towards a male intruder (two files only sniffing). Two files hold the
authors' rSLDS weights of each unit on "latent factor 1". There are no female-intruder sessions, no
mark of the pre-intruder baseline, the intruder's entry or removal, and no raw dF/F. The behaviour
and photometry cohorts (Figures 2, 3), the slice imaging (Figures 1F, 2E, S7C-L) and the outside
RNA-seq atlas (Figure 1E) are not in it.

This notebook tests the claims that need no rSLDS fit. The rSLDS claims are in `rslds_seeds_0_1.ipynb`
and `rslds_seeds_2_3.ipynb` (the four seeds the claims file asks for do not fit in one notebook's
time limit).

| Claim | Here |
|---|---|
| methods-unit-counts | tested |
| figs4a-unilateral-no-behavior-deficit | tested |
| fig4ef-male-activity-slightly-lower | tested (baseline assumed) |
| figs4e-active-units-unchanged | tested (baseline assumed) |
| fig5bd-behavior-selective-units-reduced | tested (baseline assumed) |
| figs5e-behavior-decoding-intact | tested |
| fig7c-achw-all-units | tested |
| fig5ac-behavior-activity-male-lower | blocked, data missing (female part); male part shown |
| figs5c-peth-activity | blocked, data missing (female part); male part shown |
| fig5e-cp-tuning-shift | blocked, data missing (female sniffing) |
| fig4ef-female-activity-unchanged, fig4g-male-selective-reduced, fig4g-female-selective-and-mixed-increased, fig4d-sex-decoding-intact, fig4c-pls-sex-separation, figs7b-achw-female-slight | blocked, data missing (no female-intruder data) |
| fig1e, fig1f, fig2e, fig2g, fig2 (sniff and mating), fig3c, fig3d, fig3fh, figs3ef, figs7cd, figs7k, figs7l | blocked (data not in the dandiset; fig1e uses an outside atlas, not attempted) |
| fig6-rslds-fit-quality, fig6a, fig6c, fig6f, fig6g, fig7b | tested in the rSLDS notebooks |
| fig6e-integration-dimension-persistence | blocked, metadata insufficient (no intruder removal times); variant in the rSLDS notebooks |
| figs6f-mating-attractor-intact | blocked, data missing (female) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vmh

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
mice = vmh.load_all()
G = {m["mouse"]: m["group"] for m in mice}
M = {m["mouse"]: m for m in mice}
print(f"loaded {len(mice)} files from dandiset {vmh.DANDISET_ID} version {vmh.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{
    "mouse": m["mouse"], "group": m["group"], "path": m["path"], "frames": m["Y"].shape[0],
    "units": m["Y"].shape[1], "rate_hz": m["rate"], "duration_s": m["Y"].shape[0] / m["rate"],
    "first_behaviour_s": vmh.first_behaviour_s(m),
    **{f"{b}_bouts": len(m["bouts"].get(b, [])) for b in vmh.BEHAVIOURS},
    "rslds_weights": m["rslds_weights"] is not None, "session_start": m["session_start"][:10],
    "trace_unit": m["trace_unit"]} for m in mice])
print(info.round(1).to_string(index=False))
print("\nMice and units per group:")
print(info.groupby("group").agg(mice=("mouse", "count"), units=("units", "sum"),
                                frames=("frames", "sum")).to_string())
print("\nInterval series names per file:", sorted({k for m in mice for k in m["series"]}))
print("Sampling rate of every trace:", sorted({m["rate"] for m in mice}), "Hz; starting time",
      sorted({m["starting_time"] for m in mice}))

The group is read from the subject id (`ctrl...` and `mut...`; the keywords of eight files also say
"control" or "mutant"). The dandiset holds 4 control and 6 experimental mice, where the STAR Methods
give 5 and 7. Eight files name their behaviours `Ch1_sniffing`, `Ch1_attack` and
`Ch1_dominance_mounting`; the files of mice 510 and 500 hold only `Ch1: sniffing`, although the
paper's exemplar mice attack (Figure 6B, D). Every session but one has the same placeholder start
date, 2023-02-01.

In [ ]:
rows = []
for m in mice:
    Y = m["Y"]
    rows.append({"mouse": m["mouse"], "unit_mean_absmax": np.abs(Y.mean(0)).max(),
                 "unit_sd_min": Y.std(0, ddof=1).min(), "unit_sd_max": Y.std(0, ddof=1).max(),
                 "min": Y.min(), "max": Y.max(), "nan": int(np.isnan(Y).sum()),
                 "described_as": m["trace_description"][:60], "comments": m["trace_comments"][:60]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

Every unit of nine files has mean 0 and SD 1 over its file: the traces are z-scored over the whole
file, as eight files say. In mouse 500 the SDs are below 1 (from the table), so its trace was
z-scored over a longer recording and then cut. The files of mice 510 and 500 describe the same kind
of values as "DF/F" from "GCaMP6m"; the values are z-scores, and the paper used GCaMP8s
(STAR Methods), so that description is wrong. Units of sigma above a baseline are computed from the
z-scored trace: z is an affine map of each unit's dF/F, so (z - baseline mean) / baseline SD is the
same as the Methods' normalization of dF/F.

In [ ]:
# Behaviour intervals: onsets and offsets alternate, lie inside the recording, and do not overlap.
rows = []
for m in mice:
    T = len(m["Y"]) / m["rate"]
    for beh, b in m["bouts"].items():
        d, t = m["series"][m["bout_series"][beh]]
        rows.append({"mouse": m["mouse"], "behaviour": beh, "bouts": len(b),
                     "alternating": bool(np.all(d[::2] == 1) and np.all(d[1::2] == -1)),
                     "inside_recording": bool(b.min() >= 0 and b.max() <= T),
                     "overlapping": int((b[1:, 0] < b[:-1, 1]).sum()),
                     "dur_min_s": (b[:, 1] - b[:, 0]).min(), "dur_median_s": np.median(b[:, 1] - b[:, 0]),
                     "dur_max_s": (b[:, 1] - b[:, 0]).max(), "first_s": b[0, 0], "last_s": b[-1, 1],
                     "recording_s": T})
print(pd.DataFrame(rows).round(1).to_string(index=False))
both = {}
for m in mice:
    if "attack" in m["bouts"]:
        n = len(m["Y"])
        both[m["mouse"]] = int((vmh.frames(m["bouts"]["sniff"], n) & vmh.frames(m["bouts"]["attack"], n)).sum())
print("\nFrames annotated both sniff and attack, per mouse:", both)

Mouse 511 has only two sniffing bouts, of 22 and 40 s, against tens of short bouts in the other
mice. Its sniffing annotation is kept as it is.

In [ ]:
# Duplicates. (1) The two files of the earlier version against the ten of the pinned one.
old = vmh.load_all(vmh.EARLIER_VERSION)
for o in old:
    same = [m["mouse"] for m in mice if m["Y"].shape == o["Y"].shape and np.array_equal(m["Y"], o["Y"])]
    sb = [m["mouse"] for m in mice if same and m["mouse"] in same and
          np.array_equal(m["bouts"]["sniff"], o["bouts"]["sniff"]) and
          np.array_equal(m["rslds_weights"], o["rslds_weights"])]
    print(f"{vmh.EARLIER_VERSION} {o['path']} (subject {o['mouse']}): identical trace in {same}; "
          f"identical sniffing bouts and rSLDS weights in {sb}")
# (2) Any unit trace repeated within or across the ten files, and the largest correlation between
# population means of two mice over their common length.
cols = {}
for m in mice:
    for j in range(m["Y"].shape[1]):
        cols.setdefault(np.round(m["Y"][:, j], 6).tobytes(), []).append((m["mouse"], j))
print("unit traces that occur more than once:", sum(len(v) > 1 for v in cols.values()))
best = (0, None)
names = [m["mouse"] for m in mice]
for i in range(len(mice)):
    for j in range(i + 1, len(mice)):
        n = min(len(mice[i]["Y"]), len(mice[j]["Y"]))
        r = np.corrcoef(mice[i]["Y"][:n].mean(1), mice[j]["Y"][:n].mean(1))[0, 1]
        if abs(r) > abs(best[0]):
            best = (r, (names[i], names[j]))
print("largest correlation of population means between two mice: %.3f %s" % best)
del old, cols

The two files of version 0.240826.1647 ("ControlMouse" and "MutantMouse") are mice 510 and 500 of
the pinned version, with identical traces, sniffing bouts and rSLDS weights, so the earlier version
adds nothing and each recording is counted once. No unit trace occurs twice, and no two mice share a
recording (largest correlation of population means printed above).

### The clock of traces and annotations

The traces and the behaviour annotations are given on one clock (traces start at 0 s at 10 Hz,
interval timestamps in seconds). To check that they line up, a decoder of "attack or sniff" against
all other frames is trained on the first half of each session and tested on the second, with the
labels shifted in time. If the clocks agree, the decoder does best at a shift near zero.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

shifts = [-30, -10, -5, -2, 0, 2, 5, 10, 30]
rows = []
for m in mice:
    Y, n = m["Y"], len(m["Y"])
    lab = np.zeros(n, bool)
    for b in m["bouts"].values():
        lab |= vmh.frames(b, n)
    # frames from the first to the last annotated behaviour, in four blocks: train on 1 and 3, test on 2 and 4
    a = int(min(b[0, 0] for b in m["bouts"].values()) * m["rate"])
    z = int(max(b[-1, 1] for b in m["bouts"].values()) * m["rate"])
    block = np.arange(z - a) * 4 // (z - a)
    train = np.isin(block, [0, 2])
    r = {"mouse": m["mouse"]}
    for s in shifts:
        y = np.roll(lab, int(s * m["rate"]))[a:z]  # label of frame t is the annotation at t - s
        if y[train].mean() in (0, 1) or y[~train].mean() in (0, 1):
            r[s] = np.nan
            continue
        clf = LogisticRegression(C=0.1, max_iter=2000).fit(Y[a:z][train], y[train])
        r[s] = roc_auc_score(y[~train], clf.decision_function(Y[a:z][~train]))
    rows.append(r)
clock = pd.DataFrame(rows).set_index("mouse")
print("Test AUC of the behaviour decoder by label shift (s), within the interaction; positive shift = labels delayed:")
print(clock.round(3).to_string())
print("\nshift of the best AUC per mouse:", clock.idxmax(axis=1).to_dict())
print("best AUC per mouse:", clock.max(axis=1).round(3).to_dict())

Within the interaction (first to last annotated behaviour), a decoder of "any annotated behaviour"
against the frames between bouts is trained on two interleaved quarters and tested on the other
two. In mice 482, 508, 511, 440, 441, 488 and 501 it does best at a shift of 2 to 5 s (activity
following the annotation, as a calcium signal would) and worse at 30 s in either direction, which is
consistent with traces and annotations sharing a clock. In mice 497, 510 and 500 the AUC changes
little with the shift or peaks far from zero (table), so their alignment could not be established;
they are kept, and this is said where it bears on a result.

### The pre-intruder baseline

The claims in units of sigma need the pre-intruder baseline, which the files do not mark (the
Methods say a 5-min baseline alone starts every session). The frames before the first annotated
behaviour are taken as the pre-intruder period (an assumption; see `vmh.py`). The population mean
below shows whether activity changes at the first annotated behaviour.

In [ ]:
rows = []
for m in mice:
    pm, i0 = m["Y"].mean(1), int(round(vmh.first_behaviour_s(m) * m["rate"]))
    seg = lambda a, b: pm[max(0, i0 + int(a * 10)):max(0, i0 + int(b * 10))].mean() if i0 + int(b * 10) > 0 else np.nan
    rows.append({"mouse": m["mouse"], "pre_s": i0 / 10, "baseline_used": i0 >= 70,
                 "z[-60,-10]": seg(-60, -10), "z[-10,0]": seg(-10, 0), "z[0,30]": seg(0, 30),
                 "z[30,120]": seg(30, 120)})
print("Population mean z-score in windows relative to the first annotated behaviour (s):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

Mean activity rises after the first annotated behaviour in every mouse with a baseline (table). In
mouse 482 it is already higher in the 10 s before it, which suggests the intruder entered a few
seconds before the first annotated sniff; the variant baseline ending 10 s before the first
behaviour (`gap10`) is shown alongside the two readings for that reason. Mouse 488 has only 6.2 s
before its first annotation, less than the Methods' "7sec or more", and is left out of every
sigma-based measure. Only mice 508 and 511 have more than 5 min before the first annotation.

## Claim methods-unit-counts

> CRISPRoscopy recorded 585 units from 5 control mice and 546 units from 7 experimental mice.

In [ ]:
counts = info.groupby("group").agg(mice=("mouse", "count"), units=("units", "sum"))
counts["paper_mice"] = [5, 7]
counts["paper_units"] = [585, 546]
print(counts.to_string())
print("\nunits per mouse:", info.set_index("mouse")["units"].to_dict())

**Result: not reproduced.** The dandiset holds 4 control mice with 587 units and 6 experimental mice
with 554 units, against 5 with 585 and 7 with 546 in the STAR Methods. One mouse of each group is
missing, and each group nonetheless has more units than the paper reports (2 and 8 more).

## Claim figs4a-unilateral-no-behavior-deficit

> Mice with unilateral editing and imaging (the CRISPRoscopy mice) show no deficit in sniffing or
> aggression compared with controls.

Per mouse, from the annotations, over the interaction (from the first annotated behaviour to the end
of the file; the intruder's entry is not marked). Mice 510 and 500 have no attack annotation, so
attack measures use 3 control and 5 experimental mice. The interval between attacks is the mean gap
from the end of one bout to the start of the next (onset-to-onset shown too). The "attack
probability" of panel S4A(vii) is a time course whose summary the paper does not define, and is not
computed. Time is given both in seconds and as a percentage of the interaction (the panels'
axes say "% of total time").

In [ ]:
rows = []
for m in mice:
    t0 = vmh.first_behaviour_s(m)
    T = len(m["Y"]) / m["rate"] - t0
    sn = m["bouts"]["sniff"]
    d = sn[:, 1] - sn[:, 0]
    r = {"mouse": m["mouse"], "group": m["group"], "interaction_s": T, "sniff_total_s": d.sum(),
         "sniff_pct": 100 * d.sum() / T, "sniff_bout_s": d.mean()}
    if "attack" in m["bouts"]:
        a = m["bouts"]["attack"]
        da = a[:, 1] - a[:, 0]
        r.update(attack_total_s=da.sum(), attack_pct=100 * da.sum() / T, attack_bout_s=da.mean(),
                 attack_bout_pct=100 * da.mean() / T, latency_s=a[0, 0] - t0,
                 interval_s=np.mean(a[1:, 0] - a[:-1, 1]), interval_onset_s=np.mean(np.diff(a[:, 0])),
                 attack_bouts=len(a))
    rows.append(r)
beh = pd.DataFrame(rows)
print(beh.round(2).to_string(index=False))
tests = []
for c in beh.columns[3:]:
    ctl, exp = beh[beh.group == "control"][c].dropna(), beh[beh.group == "experimental"][c].dropna()
    U, p = vmh.mw(ctl, exp)
    tests.append({"measure": c, "n_control": len(ctl), "n_exp": len(exp), "mean_control": ctl.mean(),
                  "mean_exp": exp.mean(), "exp/control": exp.mean() / ctl.mean(), "U": U, "p": p})
tests = pd.DataFrame(tests)
print("\nMann-Whitney, control against experimental:")
print(tests.round(3).to_string(index=False))
print("\nsmallest p over the measures:", round(tests.p.min(), 3),
      "; smallest attainable two-sided p with 3 and 5 mice:", round(2 / 56, 3))

**Result: reproduced (resting on the assumed start of the interaction for the latency).** No measure
differs between groups at p < 0.05 (smallest p printed above, against an attainable minimum of 0.036
for the attack measures). The experimental mice attack more, not less: their mean total attack time
and number of bouts are higher than the controls' (table). The attack probability of S4A(vii) is not
tested. The number of attack bouts, which the claim does not list, is the one measure with p < 0.05
(higher in experimental mice).

## Claim fig4ef-male-activity-slightly-lower

> During interactions with a male intruder, the activity of all VMHvl Esr1 units (sigma above
> pre-intruder baseline, and z-scored) is slightly but significantly lower in experimental than in
> control mice.

Per unit, mean activity over the first 1 or 2 min from the first annotated behaviour, in units of
sigma (baseline readings `all` and `7s`, variant `gap10`) and z-scored. Tests under the two readings
of "nested": pooled units (`p_mw_pooled`, `p_ks_pooled`) and per-mouse means or medians
(`p_mw_mice`, `p_mw_mouse_medians`).

In [ ]:
def activity_table(select=None):
    rows = []
    for minutes in (1, 2):
        for base in ("all", "7s", "gap10"):
            vs, vz = {}, {}
            for m in mice:
                s = vmh.sigma_trace(m, base)
                if s is None:
                    continue
                w = vmh.window_frames(m, minutes)
                v = s[w].mean(0)
                keep = np.ones(len(v), bool) if select is None else select(v)
                vs[m["mouse"]] = v[keep]
                vz[m["mouse"]] = m["Y"][w].mean(0)[keep]
            a, b = vmh.nested_tests(vs, G), vmh.nested_tests(vz, G)
            rows.append({"minutes": minutes, "baseline": base, **a,
                         "z_mean_c": b["pooled_mean_c"], "z_mean_e": b["pooled_mean_e"],
                         "z_p_mw_pooled": b["p_mw_pooled"], "z_p_mw_mice": b["p_mw_mice"],
                         "z_p_ks_pooled": b["p_ks_pooled"], "z_p_mw_mouse_medians": b["p_mw_mouse_medians"]})
    return pd.DataFrame(rows)

act = activity_table()
print(act.round(4).to_string(index=False))
print("\nper-mouse mean sigma (baseline all, 1 min):")
per = {m["mouse"]: vmh.sigma_trace(m, "all")[vmh.window_frames(m, 1)].mean(0).mean()
       for m in mice if vmh.sigma_trace(m, "all") is not None}
print({k: round(v, 2) for k, v in per.items()})
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for i, minutes in enumerate((1, 2)):
    for g, col in (("control", "k"), ("experimental", "r")):
        v = np.concatenate([vmh.sigma_trace(m, "all")[vmh.window_frames(m, minutes)].mean(0)
                            for m in mice if m["group"] == g and vmh.sigma_trace(m, "all") is not None])
        ax[i].plot(np.sort(v), np.linspace(0, 1, len(v)), col, label=f"{g} ({len(v)} units)")
        print(f"{minutes} min {g}: sigma range {v.min():.2f} to {v.max():.2f}")
    ax[i].set(xlabel="mean activity (sigma, baseline all)", ylabel="ECDF", title=f"first {minutes} min")
    ax[i].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Result: reproduced, under the pooled reading of "nested" and resting on the assumed baseline and
start of interaction.** With units pooled, experimental activity in sigma is lower than control by
both the Mann-Whitney and the KS test at p < 0.05 under every baseline reading and both windows
(table; for example 0.53 against 0.95 sigma, p = 0.002 and KS p < 0.001, baseline `all`, 1 min).
On per-mouse means or medians (4 control, 5 experimental mice) no comparison reaches p < 0.05. The
difference is not slight: the experimental mean is about half the control mean. The z-scored
activity is also lower in experimental mice when pooled (KS p < 0.01 in both windows), but its
Mann-Whitney test does not reach p < 0.05.

## Claim figs4e-active-units-unchanged

> Among male-responsive units (at least 2 sigma above baseline), mean activity does not differ
> between control and experimental mice.

In [ ]:
active = activity_table(select=lambda v: v >= vmh.ACTIVE_SIGMA)
print(active[["minutes", "baseline", "n_mice_c", "n_mice_e", "n_units_c", "n_units_e", "pooled_mean_c",
              "pooled_mean_e", "mouse_mean_c", "mouse_mean_e", "p_mw_pooled", "p_mw_mice"]].round(4).to_string(index=False))

**Result: reproduced, resting on an assumption: "nested" read as a test across mice.** On per-mouse
means the nested Mann-Whitney test finds no difference under any baseline or window (p from 0.19 to
0.90). With units pooled, the readings disagree: no difference with the `all` baseline (p = 0.09 at
1 min and 0.76 at 2 min), but control higher at p < 0.001 with the `7s` baseline, where the narrow
baseline inflates sigma (control mean 12.8 against experimental 8.5). The paper ran its nested tests
in GraphPad Prism (STAR Methods, Statistical analysis), whose nested analyses treat the animal as the
unit of replication; the per-mouse reading is adopted on that basis.

## Claim fig5bd-behavior-selective-units-reduced

> During male interactions, sniff-selective units are reduced from 8.9 +/- 3% to 3.9 +/- 2% and
> attack-selective units from 6 +/- 0.9% to 1.8 +/- 0.7% in experimental mice; the mixed-selectivity
> fraction is reduced by about 22%, and units active during male-directed behaviors fall from 36.2%
> to 22.4%; female-directed selective fractions are similar.

Mice with both sniffing and attack annotations and a baseline: control 482, 508, 511; experimental
440, 441, 497, 501. Readings (a) and (b) of "active during all male-directed behaviors" give the same
number (sniff-selective + attack-selective + mixed), so one is shown.

In [ ]:
expected = {"sniff_sel_e": (3.9, 2), "sniff_sel_c": (8.9, 3), "att_sel_e": (1.8, 0.7), "att_sel_c": (6, 0.9),
            "active_c": (36.2, 3.62), "active_e": (22.4, 2.24), "mixed_reduction": (22, 2.2)}
sel_rows, per_mouse = [], []
for base in ("all", "7s", "gap10"):
    per = []
    for m in mice:
        if "attack" not in m["bouts"]:
            continue
        s = vmh.sigma_trace(m, base)
        if s is None:
            continue
        n = len(s)
        sn = s[vmh.frames(m["bouts"]["sniff"], n)].mean(0)
        at = s[vmh.frames(m["bouts"]["attack"], n)].mean(0)
        per.append({"baseline": base, "mouse": m["mouse"], "group": m["group"], "units": len(sn),
                    "sniff_sel": int(((sn >= 2) & (at < 2)).sum()), "att_sel": int(((at >= 2) & (sn < 2)).sum()),
                    "mixed": int(((sn >= 2) & (at >= 2)).sum())})
    per = pd.DataFrame(per)
    per["active"] = per.sniff_sel + per.att_sel + per.mixed
    per_mouse.append(per)
    for how in ("pooled", "mouse_mean"):
        r = {"baseline": base, "fractions": how}
        for g, d in per.groupby("group"):
            for c in ("sniff_sel", "att_sel", "mixed", "active"):
                r[f"{c}_{g[0]}"] = (100 * d[c].sum() / d.units.sum() if how == "pooled"
                                    else (100 * d[c] / d.units).mean())
        r["mixed_reduction"] = 100 * (1 - r["mixed_e"] / r["mixed_c"])
        r["within"] = sum(abs(r[k] - v) <= t for k, (v, t) in expected.items())
        sel_rows.append(r)
per_mouse = pd.concat(per_mouse)
print(per_mouse.to_string(index=False))
sel = pd.DataFrame(sel_rows)
print("\nPercentages (c control, e experimental); 'within' counts the 7 expected values inside tolerance:")
print(sel.round(1).to_string(index=False))
for _, r in sel.iterrows():
    print(r.baseline, r.fractions, "outside tolerance:",
          [k for k, (v, t) in expected.items() if abs(r[k] - v) > t])
for base in ("all", "7s"):
    d = per_mouse[per_mouse.baseline == base]
    for c in ("sniff_sel", "att_sel", "active"):
        f = 100 * d[c] / d.units
        print(base, c, "Mann-Whitney across mice p = %.3f" % vmh.mw(f[d.group == "control"], f[d.group == "experimental"])[1])

**Result: partially reproduced, resting on the assumed baseline.** The directions hold under every
reading: sniff-selective, attack-selective and active fractions are all lower in experimental mice
(table). The values do not all fall inside tolerance under any one reading. With the `all` baseline
and pooled fractions, control sniff-selective 9.6% and attack-selective 6.0% match the paper, and
experimental attack-selective 2.4% is inside 1.8 +/- 0.7%, but experimental sniff-selective (1.7%)
is below 3.9 +/- 2%, the active fractions (28.8% and 11.4%) are below 36.2% and 22.4%, and the
mixed fraction falls by 45%, not 22%. With the `7s` baseline more units pass 2 sigma and the control
values exceed the paper's. The female-directed fractions are not in the dandiset.

## Claim figs5e-behavior-decoding-intact

> Linear decoders distinguish attack from sniffing equally well in control and experimental mice.

Per mouse, frame-wise linear SVM on balanced attack and sniff frames, stratified five-fold
cross-validation, 20 repeats (STAR Methods; claims file). Mice with attack annotations: 3 control,
5 experimental. Shuffled decoders reassign the bouts' labels at random (20 repeats here, against the
paper's 1,000, a standard choice to keep the run short; the shuffle does not decide the outcome).
The bout-wise split is a variant that keeps each bout in one fold.

In [ ]:
rng = np.random.default_rng(5)
rows = []
for m in mice:
    if "attack" not in m["bouts"]:
        continue
    r = {"mouse": m["mouse"], "group": m["group"]}
    for C in (0.1, 1.0, 10.0):
        r[f"acc_C{C:g}"] = vmh.decode_attack_vs_sniff(m, rng, C=C)
    r["shuffled_C1"] = vmh.decode_attack_vs_sniff(m, rng, C=1.0, shuffle=True)
    r["by_bout_C1"] = vmh.decode_attack_vs_sniff(m, rng, C=1.0, by_bout=True)
    r["by_bout_shuffled_C1"] = vmh.decode_attack_vs_sniff(m, rng, C=1.0, by_bout=True, shuffle=True)
    rows.append(r)
dec = pd.DataFrame(rows)
print(dec.round(3).to_string(index=False))
print()
for c in dec.columns[2:]:
    a, b = dec[dec.group == "control"][c], dec[dec.group == "experimental"][c]
    print(f"{c:22s} control {a.mean():.3f}  experimental {b.mean():.3f}  difference {a.mean() - b.mean():+.3f}"
          f"  Mann-Whitney p = {vmh.mw(a, b)[1]:.3f}")
for g in ("control", "experimental"):
    d = dec[dec.group == g]
    print(f"{g}: real against shuffled (C=1), Mann-Whitney across mice p = {vmh.mw(d.acc_C1, d.shuffled_C1)[1]:.3f}")
print("smallest attainable two-sided p with 3 and 5 mice:", round(2 / 56, 3))

**Result: not reproduced.** With the paper's frame-wise split, every control mouse decodes better
than every experimental mouse at all three values of C, and the Mann-Whitney test across mice finds
the difference at p = 0.036, the smallest p 3 and 5 mice allow (table). The difference is small
(0.06 to 0.07 in accuracy). The frame-wise split does not separate frames of the same bout, so decoders
trained on shuffled bout labels score almost as well as on real labels (0.988 and 0.922 against 0.993
and 0.934). With whole bouts held out (a variant), accuracy falls to 0.78 in control and 0.53 in
experimental mice, near the shuffled 0.54 and 0.49, and the groups do not differ at p < 0.05.

## Claim fig7c-achw-all-units

> Across all VMHvl Esr1 units, the mean ACHW during the first 1-2 minutes of male-male interaction is
> reduced in experimental mice (by about 50.9%), less than for first-dimension units.

ACHW per unit over the first 1 or 2 min from the first annotated behaviour. The autocorrelation
is unchanged by an affine map of the trace, so the claims file's z-scored and sigma readings give
the same ACHW, and the sigma reading needs no baseline. With one male intruder per file, the "mean
over intruder trials" reading is the same as the single autocorrelation. All 10 mice are used.
The whole recording is shown as a variant.

In [ ]:
rows = []
achw_values = {}
for label, sel_frames in (("1 min", lambda m: vmh.window_frames(m, 1)),
                          ("2 min", lambda m: vmh.window_frames(m, 2)),
                          ("whole file (variant)", lambda m: np.arange(len(m["Y"])))):
    v = {m["mouse"]: vmh.achw(m["Y"][sel_frames(m)]) for m in mice}
    achw_values[label] = v
    r = vmh.nested_tests(v, G)
    rows.append({"window": label, **r,
                 "reduction_pooled_pct": 100 * (1 - r["pooled_mean_e"] / r["pooled_mean_c"]),
                 "reduction_mice_pct": 100 * (1 - r["mouse_mean_e"] / r["mouse_mean_c"]),
                 "undefined": int(sum(np.isnan(x).sum() for x in v.values()))})
ach = pd.DataFrame(rows)
print(ach.round(4).to_string(index=False))
print("\nper-mouse mean ACHW (s), 2 min:", {k: round(np.nanmean(x), 2) for k, x in achw_values["2 min"].items()})
print("expected reduction 50.9 +/- 5.09 %")
fig, ax = plt.subplots(figsize=(4.5, 3.2))
for g, col in (("control", "k"), ("experimental", "r")):
    x = np.concatenate([achw_values["2 min"][k] for k in achw_values["2 min"] if G[k] == g])
    ax.plot(np.sort(x), np.linspace(0, 1, len(x)), col, label=g)
    print(f"{g}: ACHW range (2 min) {np.nanmin(x):.1f} to {np.nanmax(x):.1f} s")
ax.set(xlabel="ACHW (s), first 2 min", ylabel="ECDF"); ax.legend(); plt.tight_layout(); plt.show()

**Result: not reproduced.** Mean ACHW over the first 1 or 2 min is shorter in experimental mice, and
significantly so with units pooled (Mann-Whitney and KS p < 0.001), but not across mice (p = 0.26
and 0.11). The reduction is 15 to 19% (pooled or per-mouse means), not the paper's 50.9 +/- 5.1%.
Over the whole recording (a variant, not the claim's window) the reduction is about 45%.

## Claims blocked for missing data, with their male part

fig5ac-behavior-activity-male-lower and figs5c-peth-activity need the female-intruder sessions,
which the dandiset does not hold; their male-directed parts are shown here and do not decide an
outcome.

In [ ]:
rows = []
for base in ("all", "7s"):
    for b in ("sniff", "attack"):
        vs = {}
        for m in mice:
            s = vmh.sigma_trace(m, base)
            if s is None or b not in m["bouts"]:
                continue
            vs[m["mouse"]] = s[vmh.frames(m["bouts"][b], len(s))].mean(0)
        rows.append({"baseline": base, "behaviour": b, **vmh.nested_tests(vs, G)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
rows = []
for pre in (2, 3):
    for b in ("sniff", "attack"):
        v = {}
        for m in mice:
            if b not in m["bouts"]:
                continue
            Y, n = m["Y"], len(m["Y"])
            per_bout = []
            for on, _ in m["bouts"][b]:
                i = int(round(on * 10))
                if i - pre * 10 < 0 or i + 150 > n:
                    continue
                per_bout.append(Y[i:i + 150].mean(0) - Y[i - pre * 10:i].mean(0))
            v[m["mouse"]] = np.mean(per_bout, axis=0)
        rows.append({"pre_bout_baseline_s": pre, "behaviour": b, **vmh.nested_tests(v, G)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## Summary

| Claim | Outcome |
|---|---|
| methods-unit-counts | not reproduced: 4 and 6 mice, 587 and 554 units, against 5 and 7, 585 and 546 |
| figs4a-unilateral-no-behavior-deficit | reproduced (latency rests on an assumption) |
| fig4ef-male-activity-slightly-lower | reproduced with units pooled (assumed baseline); not across mice |
| figs4e-active-units-unchanged | reproduced (assumed: test across mice; assumed baseline) |
| fig5bd-behavior-selective-units-reduced | partially reproduced (directions hold, values outside tolerance) |
| figs5e-behavior-decoding-intact | not reproduced: control decodes better, p = 0.036 |
| fig7c-achw-all-units | not reproduced: reduction 15 to 19%, not 50.9% |
| fig5ac, figs5c, fig5e and the female-intruder claims | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset gives z-scored traces only and marks neither
the pre-intruder baseline nor the intruder's entry, so the baseline is taken as the frames before the
first annotated behaviour, and the interaction as starting at that behaviour. One control and one
experimental mouse are not in the dandiset, and two mice have no attack annotation, so tests across
mice use 3 to 5 mice per group. "Nested" tests are not defined in the paper; both readings are shown.

What was found about the dataset. One mouse per group is missing; units per group differ from the
Methods' counts; the files of mice 510 and 500 describe z-scores as "DF/F" from "GCaMP6m" (the paper
used GCaMP8s) and hold only sniffing annotations; the baseline, the intruder's entry and removal and
the female-intruder sessions are absent; session start dates are placeholders; the earlier published
version's two files duplicate mice 510 and 500.